In [ ]:
"""
TASK 1
"""

import requests
import pandas as pd
import matplotlib.pyplot as plt

from CLIENT_INFO import client_id

# Frost API endpoint and parameters for daily mean air temperature
endpoint = "https://frost.met.no/observations/v0.jsonld"
parameters = {
    "sources": "SN17850",  # Weather station ID for Ås
    "elements": "mean(air_temperature P1D)",  # Daily mean air temperature
    "referencetime": "2025-01-01/2025-12-31",  # Full year of 2025
}

# Send request to Frost API using client ID for authentication
response = requests.get(endpoint, params=parameters, auth=(client_id, ""))
data = response.json()

# Extract observations from the API response
obs = data["data"]

# Build a list of rows containing time and temperature values
rows = []
for item in obs:
    time = item["referenceTime"]          # Timestamp of the observation
    temp = item["observations"][0]["value"]  # Temperature value
    rows.append([time, temp])

# Create a DataFrame with time and temperature columns
df = pd.DataFrame(rows, columns=["time", "temperature"])

# Convert time column to datetime format
df["time"] = pd.to_datetime(df["time"])

# Plot daily mean temperature throughout the year
plt.figure(figsize=(12,5))
plt.plot(df["time"], df["temperature"], color="royalblue")

plt.title("Daily Mean Temperature – Ås (2025)")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.grid(True)

plt.tight_layout()
plt.show()

# Create a summary table with basic statistics
summary = {
    "Mean": df["temperature"].mean(),
    "Median": df["temperature"].median(),
    "Min": df["temperature"].min(),
    "Max": df["temperature"].max()
}

summary_df = pd.DataFrame([summary])
print(summary_df)


In [ ]:
"""
TASK 2
"""
import requests
import pandas as pd
from matplotlib import pyplot as plt
from  CLIENT_INFO import client_id

#Gotten from "https://frost.met.no/python_example.html"
endpoint = 'https://frost.met.no/observations/v0.jsonld'
parameters = {
    'sources': 'SN17850',
    'elements': 'sum(precipitation_amount P1D)',
    'referencetime': '2025-01-01/2025-12-31',
}
# Issue an HTTP GET request
r = requests.get(endpoint, parameters, auth=(client_id,''))  
# Extract JSON data
json = r.json()

if r.status_code == 200:
    data = json['data']
    print('Data retrieved from frost.met.no!')
else:
    print('Error! Returned status code %s' % r.status_code)
    print('Message: %s' % json['error']['message'])
    print('Reason: %s' % json['error']['reason'])


df = pd.json_normalize(
    data,
    record_path="observations",
    meta=["referenceTime", "sourceId"],
)

df = df[df["timeOffset"]  == "PT6H"].reset_index(drop=True)
print(df.head())

plt.plot(df["value"])
plt.xlabel("Days")
plt.ylabel(f"Precipation ({df['unit'][0]})")
plt.show()

In [ ]:
"""
TASK 3
"""

import requests
import pandas as pd
import matplotlib.pyplot as plt

from CLIENT_INFO import client_id

# Frost API endpoint
endpoint = "https://frost.met.no/observations/v0.jsonld"

# Parameters for 2025 temperatures
parameters_2025 = {
    "sources": "SN17850",  # Weather station ID for Ås
    "elements": "mean(air_temperature P1D)",  # Daily mean air temperature
    "referencetime": "2025-01-01/2025-12-31",
}

# Parameters for 1925 temperatures
parameters_1925 = {
    "sources": "SN17850",  # Same station, 100 years earlier
    "elements": "mean(air_temperature P1D)",
    "referencetime": "1925-01-01/1925-12-31",
}

# Request 2025 data
response_2025 = requests.get(endpoint, params=parameters_2025, auth=(client_id, ""))
data_2025 = response_2025.json()
obs_2025 = data_2025["data"]

# Request 1925 data
response_1925 = requests.get(endpoint, params=parameters_1925, auth=(client_id, ""))
data_1925 = response_1925.json()
obs_1925 = data_1925["data"]

# Build DataFrame for 2025
rows_2025 = []
for item in obs_2025:
    time = item["referenceTime"]          # Timestamp of the observation
    temp = item["observations"][0]["value"]  # Temperature value
    rows_2025.append([time, temp])

df_2025 = pd.DataFrame(rows_2025, columns=["time", "temperature"])
df_2025["time"] = pd.to_datetime(df_2025["time"])

# Build DataFrame for 1925
rows_1925 = []
for item in obs_1925:
    time = item["referenceTime"]
    temp = item["observations"][0]["value"]
    rows_1925.append([time, temp])

df_1925 = pd.DataFrame(rows_1925, columns=["time", "temperature"])
df_1925["time"] = pd.to_datetime(df_1925["time"])

# Normalize dates to day-of-year so the curves can be compared properly
df_2025["dayofyear"] = df_2025["time"].dt.dayofyear
df_1925["dayofyear"] = df_1925["time"].dt.dayofyear

# Plot both temperature curves in the same figure
plt.figure(figsize=(12, 5))
plt.plot(df_2025["dayofyear"], df_2025["temperature"], color="royalblue", label="2025")
plt.plot(df_1925["dayofyear"], df_1925["temperature"], color="darkred", label="1925")

plt.title("Daily Mean Temperature – Ås (2025 vs 1925)")
plt.xlabel("Day of Year")
plt.ylabel("Temperature (°C)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# Create summary table for both years
summary = {
    "Mean_2025": df_2025["temperature"].mean(),
    "Median_2025": df_2025["temperature"].median(),
    "Min_2025": df_2025["temperature"].min(),
    "Max_2025": df_2025["temperature"].max(),

    "Mean_1925": df_1925["temperature"].mean(),
    "Median_1925": df_1925["temperature"].median(),
    "Min_1925": df_1925["temperature"].min(),
    "Max_1925": df_1925["temperature"].max()
}

summary_df = pd.DataFrame([summary])
print(summary_df)


In [ ]:
"""
TASK 4
"""
import requests
import pandas as pd
from  CLIENT_INFO import client_id
import yaml

def heatwaves(scources = "SN17850", year = "2025") -> None:
    #Inspired from "https://frost.met.no/python_example.html"
    endpoint = 'https://frost.met.no/observations/v0.jsonld'
    parameters = {
        'sources': scources,
        'elements': 'max(air_temperature P1D)',
        'referencetime': f'{year}-01-01/{year}-12-31',
    }
    # Issue an HTTP GET request
    r = requests.get(endpoint, parameters, auth=(client_id,''))  
    # Extract JSON data
    json = r.json()

    if r.status_code == 200:
        data = json['data']
        print('Data retrieved from frost.met.no!')
    else:
        print('Error! Returned status code %s' % r.status_code)
        print('Message: %s' % json['error']['message'])
        print('Reason: %s' % json['error']['reason'])
        return


    df = pd.json_normalize(
        data,
        record_path="observations",
        meta=["referenceTime", "sourceId"],
    )
    df["referenceTime"] = pd.to_datetime(df.referenceTime).dt.date
    df = df[df["timeOffset"]  == "PT0H"].reset_index(drop=True)
    
    #Had to collect station name from scources link rather than observations
    endpoint_sc = 'https://frost.met.no/sources/v0.jsonld'
    st_name = requests.get(
        endpoint_sc, params = {"ids":scources}, auth=(client_id,"")
    )
    st_name = st_name.json()["data"][0]["name"]
    
    deg27 = df["value"] >= 27 #Makes a boolean per day to check if  deg is more than 27

    run_id = (deg27 != deg27.shift()).cumsum() #Creates an per period id, new period each time it changes from more than 27 to less

    #Nicely formmats the data of the possible heatwaves
    runs = (df[deg27]
        .groupby(run_id)
        .agg(
                dates=("referenceTime", lambda s: f"{s.iloc[0]} - {s.iloc[-1]}"),
                consecutive_days = ('value', 'count'),
            ))
    print(runs)
    dates = (runs[runs["consecutive_days"] >= 5].reset_index(drop=True))
    date_list = (dates["dates"]).to_list()

    heatwave_summmary = {
        "dates": date_list,
        "station_id": scources,
        "station_name": st_name,
        "year": year
    }

    with open("Heatwaves_" + scources + "_" + year + "_.yml", "w") as file:
        yaml.dump(heatwave_summmary, file, sort_keys=False)
    print("CODE FINISHED:  yaml file created")

heatwaves(year=2025)

In [ ]:
"""
TASK 5
"""

import requests
import pandas as pd
import matplotlib.pyplot as plt

# Correct client ID
from CLIENT_INFO import client_id

# Frost API endpoint and parameters for daily mean air temperature
endpoint = "https://frost.met.no/observations/v0.jsonld"
parameters = {
    "sources": "SN17850",  # Weather station ID for Ås
    "elements": "mean(air_temperature P1D)",  # Daily mean air temperature
    "referencetime": "2025-01-01/2025-12-31",
}

# Send request to Frost API using client ID for authentication
response = requests.get(endpoint, params=parameters, auth=(client_id, ""))
data = response.json()

# Extract observations from the API response
obs = data["data"]

# Build a list of rows containing time and temperature values
rows = []
for item in obs:
    time = item["referenceTime"]          # Timestamp of the observation
    temp = item["observations"][0]["value"]  # Temperature value
    rows.append([time, temp])

# Create a DataFrame with time and temperature columns
df = pd.DataFrame(rows, columns=["time", "temperature"])

# Convert time column to datetime format
df["time"] = pd.to_datetime(df["time"])

# --- Task 5 requirement: function for 7-day moving average ---
def moving_average(series, window=7):
    """
    Calculate a moving average over a given window size.

    Args:
        series (pd.Series): Temperature values.
        window (int): Number of days for the moving average.

    Returns:
        pd.Series: Moving average values.
    """
    return series.rolling(window=window).mean()

# Calculate the 7-day moving average for 2025 temperatures
df["moving_avg"] = moving_average(df["temperature"], window=7)

# Plot daily temperature as points and moving average as a line
plt.figure(figsize=(12, 5))

# Daily temperature as scatter points
plt.scatter(df["time"], df["temperature"], color="royalblue", s=10, label="Daily Temperature")

# Moving average as line
plt.plot(df["time"], df["moving_avg"], color="darkred", linewidth=2, label="7-Day Moving Average")

plt.title("Daily Temperature and 7-Day Moving Average – Ås (2025)")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()
